# OpenWebUI-Compatible API Client (System Prompt + User Query + Optional Document Upload)

This notebook implements a minimal, **from-first-principles** client for an **OpenAI-compatible** API, such as **OpenWebUI**. You provide an **API base URL**, an **API key**, a **model name**, a **system prompt**, and a **user query**. Optionally, upload a document; the client will **try** `/v1/files` upload if supported, or fall back to **in-message context** (truncated) if not.

**Key features**
- Works against OpenAI-style endpoints: `/v1/chat/completions` (primary) and `/v1/files` (optional).
- Supports TXT, PDF, and DOCX ingestion (via lightweight libraries); other files fall back to metadata.

## 0) Environment & Dependencies

In [1]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install requests pdfminer.six python-docx

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
In Colab: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.6 MB/s eta 0:00:00


## 1) Imports & Helpers

In [2]:
import os, io, json, time, textwrap
import requests
from getpass import getpass

# Optional file parsing libs
from pdfminer.high_level import extract_text as pdf_extract_text
from docx import Document as DocxDocument

# Colab upload helper (no-op outside Colab)
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def pretty_print_json(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))

def clamp(s: str, max_chars: int = 12000):
    s = s or ""
    return s if len(s) <= max_chars else s[:max_chars] + f"\n...[truncated {len(s)-max_chars} chars]"

## 2) Configure Your API and Query

Fill in your OpenWebUI (or compatible) **API base URL**, **API key**, and **model**. Then set your **system prompt** and **user query**. You can adjust the timeout and temperature as needed.

In [3]:
API_BASE = "https://ursinus.ai/api/v1"  # e.g., "http://<host>:<port>/v1" or "https://your.domain/v1"
API_KEY  = getpass("Enter your OpenWebUI API Key")        # put your key or leave empty if your server does not require it
MODEL    = "gpt-3.5-turbo"                  # replace with your deployed model name
TIMEOUT  = 120                          # seconds
TEMP     = 0.2

SYSTEM_PROMPT = "You are a helpful assistant. Answer concisely and cite any assumptions."
USER_QUERY    = "Summarize the attached document in 5 bullet points."

# Basic validation
assert isinstance(API_BASE, str) and API_BASE.strip(), "Please set API_BASE"
assert isinstance(MODEL, str) and MODEL.strip(), "Please set MODEL"
print("Configured. API_BASE=", API_BASE, "| MODEL=", MODEL)

Enter your OpenWebUI API Key··········
Configured. API_BASE= https://ursinus.ai/api/v1 | MODEL= gpt-3.5-turbo


## 3) (Optional) Upload a Document

Run the next cell to upload a file (TXT/PDF/DOCX recommended). If not in Colab, set `LOCAL_FILEPATH` to an existing file path.

In [ ]:
LOCAL_FILEPATH = ""  # set path on disk if not using the widget

uploaded = {}
if IN_COLAB:
    print("Use the dialog to upload a document (optional). Cancel to skip.")
    try:
        uploaded = files.upload()
        if uploaded:
            LOCAL_FILEPATH = list(uploaded.keys())[0]
            print("Uploaded:", LOCAL_FILEPATH, "| size:", len(uploaded[LOCAL_FILEPATH]), "bytes")
    except Exception as e:
        print("Upload skipped or failed:", e)

if LOCAL_FILEPATH:
    print("Selected document:", LOCAL_FILEPATH)
else:
    print("No document selected — proceeding without an attachment.")

Use the dialog to upload a document (optional). Cancel to skip.


No document selected — proceeding without an attachment.


## 4) Parse Document to Text (for Context Fallback)

We try to extract text for TXT/PDF/DOCX. If another type, we include only filename and size.

In [ ]:
def read_text_from_path(path: str) -> dict:
    meta = {"path": path, "ok": False, "text": "", "error": None}
    try:
        ext = os.path.splitext(path)[1].lower()
        if ext in [".txt", ""]:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                meta["text"] = f.read()
                meta["ok"] = True
        elif ext == ".pdf":
            meta["text"] = pdf_extract_text(path) or ""
            meta["ok"] = True
        elif ext == ".docx":
            doc = DocxDocument(path)
            meta["text"] = "\n".join(p.text for p in doc.paragraphs)
            meta["ok"] = True
        else:
            # Fallback: binary length
            sz = os.path.getsize(path)
            meta["text"] = f"[Unsupported file type {ext}; only metadata available. Size={sz} bytes.]"
            meta["ok"] = True
    except Exception as e:
        meta["error"] = str(e)
    return meta

DOC_META = None
if LOCAL_FILEPATH:
    DOC_META = read_text_from_path(LOCAL_FILEPATH)
    print("Parsed:", DOC_META["ok"], "| chars:", len(DOC_META.get("text","")))
    if DOC_META.get("error"):
        print("Parse error:", DOC_META["error"])
else:
    print("No document to parse.")

Parsed: True | chars: 1898


## 5) HTTP Helpers for OpenAI-Compatible APIs

In [ ]:
def headers_with_auth(api_key: str | None):
    h = {"Content-Type": "application/json"}
    if api_key:
        h["Authorization"] = f"Bearer {api_key}"
    return h

def post_json(url, payload, api_key=None, timeout=60):
    resp = requests.post(url, headers=headers_with_auth(api_key), json=payload, timeout=timeout)
    return resp

def post_multipart(url, files, data=None, api_key=None, timeout=60):
    hdrs = {}
    if api_key:
        hdrs["Authorization"] = f"Bearer {api_key}"
    resp = requests.post(url, headers=hdrs, files=files, data=data or {}, timeout=timeout)
    return resp

## 6) Try Upload via `/v1/files` (Graceful Fallback)

We first try to upload the document to the server using `/v1/files` with a `purpose`. If that endpoint is not supported (404/501), we proceed by injecting the document text into the user message.

In [ ]:
uploaded_file_id = None
files_api_supported = False

if LOCAL_FILEPATH:
    try:
        url_files = API_BASE.rstrip("/") + "/files"
        with open(LOCAL_FILEPATH, "rb") as fh:
            resp = post_multipart(
                url_files,
                files={"file": (os.path.basename(LOCAL_FILEPATH), fh)},
                data={"purpose": "assistants"},
                api_key=API_KEY,
                timeout=TIMEOUT
            )
        if resp.ok:
            files_api_supported = True
            data = resp.json()
            uploaded_file_id = data.get("id")
            print("Files API success. file_id:", uploaded_file_id)
        else:
            print("Files API not available or failed:", resp.status_code, resp.text[:400])
    except requests.exceptions.RequestException as e:
        print("Files API request exception:", e)
else:
    print("No file provided; skipping /v1/files.")

No file provided; skipping /v1/files.


## 7) Build Chat Messages

If `/v1/files` succeeded, we include a reference note in the prompt. Otherwise we **inline** document text (truncated) under a delimiter.

In [ ]:
INLINE_DOC_MAX = 12000  # characters

messages = [{"role": "system", "content": SYSTEM_PROMPT}]

user_content = USER_QUERY

if LOCAL_FILEPATH and uploaded_file_id:
    user_content += (
        f"\n\n[Attached file id: {uploaded_file_id}. Please use any server-side retrieval tools if available.]"
    )
elif LOCAL_FILEPATH and DOC_META:
    doc_text = clamp(DOC_META.get("text",""), max_chars=INLINE_DOC_MAX)
    user_content += (
        "\n\n----- BEGIN DOCUMENT CONTEXT (truncated) -----\n"
        + doc_text +
        "\n----- END DOCUMENT CONTEXT -----"
    )

messages.append({"role": "user", "content": user_content})
pretty_print_json({"messages": messages})

{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant. Answer concisely and cite any assumptions."
    },
    {
      "role": "user",
      "content": "Summarize the attached document in 5 bullet points."
    }
  ]
}


## 8) Invoke `/v1/chat/completions`

In [ ]:
payload = {
    "model": MODEL,
    "messages": messages,
    "temperature": TEMP,
    "stream": False
}

url_chat = API_BASE.rstrip("/") + "/chat/completions"
print("POST", url_chat)
try:
    resp = post_json(url_chat, payload, api_key=API_KEY, timeout=TIMEOUT)
    print("Status:", resp.status_code)
    if resp.ok:
        data = resp.json()
        pretty_print_json(data)
        # Extract assistant text (OpenAI-style schema)
        try:
            content = data["choices"][0]["message"]["content"]
        except Exception:
            content = None
        print("\n--- ASSISTANT REPLY ---\n")
        print(content or "[no content returned]")
        RESULT = {"request": payload, "response": data}
    else:
        print("Error body:", resp.text[:1000])
        RESULT = {"request": payload, "response": {"status": resp.status_code, "text": resp.text}}
except requests.exceptions.RequestException as e:
    print("Request exception:", e)
    RESULT = {"request": payload, "response": {"exception": str(e)}}

POST https://ursinus.ai/api/v1/chat/completions
Status: 200
{
  "id": "chatcmpl-CTXLsw6IyvQNH7U9bow45ndPN5P3V",
  "object": "chat.completion",
  "created": 1761155752,
  "model": "gpt-3.5-turbo-0125",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "- Hidden Markov model used for object recognition in images\n- Hidden Markov model applied to robot localization in a maze with heat map output\n- Hidden Markov model for bull vs bear market detection\n- Hidden Markov model for ECG artifact detection\n- Hidden Markov model for speech-to-text conversion from a wav file",
        "refusal": null,
        "annotations": []
      },
      "logprobs": null,
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 429,
    "completion_tokens": 66,
    "total_tokens": 495,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,


## 9) Save Artifacts & (Colab) Download

We persist the request/response JSON and, if provided, the parsed document text.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

with open("artifacts/chat_request_response.json", "w", encoding="utf-8") as f:
    json.dump(RESULT, f, indent=2, ensure_ascii=False)

if DOC_META:
    with open("artifacts/document_text_preview.txt", "w", encoding="utf-8") as f:
        f.write(DOC_META.get("text","")[:20000])

print("Saved:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        if IN_COLAB:
            files.download('artifacts.zip')  # type: ignore
        else:
            print("Artifacts ZIP created at ./artifacts.zip")
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Download helper note:', e)

## 10) Notes & Extensions

- If your server expects different endpoints or payload keys, adapt `url_chat` and `payload`.
- Some servers require `model` names like `gpt-4o-mini`, `llama3`, or a local ID. Check OpenWebUI model registry.
- For richer retrieval, implement server-specific tools (e.g., `/v1/assistants`, `/v1/responses` with `input_files`, etc.).
- Add streaming support by setting `stream=True` and iterating the response chunks.